In [1]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import phoebe
from phoebe import u as _u

# ---------------------------
# RUTAS A TUS ARCHIVOS
# ---------------------------
filepath_r = "D:/archivos_Investigación/PHOEBE}/TYC_6265_2079_flujo_r.txt"
filepath_i = "D:/archivos_Investigación/PHOEBE}/TYC_6265_2079_flujo_i.txt"

In [2]:
data_r = np.genfromtxt(filepath_r, delimiter=None, skip_header=1, encoding='latin1')
time_r = data_r[:, 0]
flux_r = data_r[:, 1]
fluxerr_r = data_r[:, 2]

data_i = np.genfromtxt(filepath_i, delimiter=None, skip_header=1, encoding='latin1')
time_i = data_i[:, 0]
flux_i = data_i[:, 1]
fluxerr_i = data_i[:, 2]


In [3]:
b = phoebe.default_binary()

# [NUEVO] - MORFOLOGÍA: Restringimos el sistema a semi-separado
# Esto obliga a que el radio de la secundaria sea igual a su lóbulo de Roche.
b.add_constraint('semidetached', component='secondary')

# Agregar datasets
b.add_dataset('lc', times=time_r, fluxes=flux_r, sigmas=fluxerr_r,
              passband='SDSS:r', dataset='lc01')

b.add_dataset('lc', times=time_i, fluxes=flux_i, sigmas=fluxerr_i,
              passband='SDSS:i', dataset='lc02')

# ---------------------------
# PARÁMETROS INICIALES 
# ---------------------------
b['period@binary'] = 3.6788 * u.day
b['t0_supconj@binary'] = time_r[0] * u.day
b['incl@binary'] = 70.0 * u.deg
b['q@binary'] = 0.5
b['teff@primary'] = 25700 * u.K
b['teff@secondary'] = 20000 * u.K
b['ecc@binary'] = 0.0

# Físicas de la estrella
b.set_value_all('atm', 'ck2004')
b.set_value_all('ld_mode', 'interp')
b.set_value_all('ld_mode_bol', 'manual')
b.set_value_all('gravb_bol', 1.0)
b.set_value_all('irrad_frac_refl_bol', 1.0)
b.set_value_all('pblum_mode', 'dataset-scaled')
b.set_value_all('irrad_method', 'horvat')

# ---------------------------
# GESTIÓN DE CONSTRAINTS
# ---------------------------
b.flip_constraint('teffratio', solve_for='teff@secondary')

# [MODIFICADO]: Al estar la secundaria amarrada a su lóbulo de Roche, 
# usamos el 'requivsumfrac' (que estimará la IA EBAI) para deducir el radio de la estrella primaria.
b.flip_constraint('requivsumfrac', solve_for='requiv@primary')


<ConstraintParameter: {requiv@primary@component} = ({requivsumfrac@binary@component} * {sma@binary@component}) - {requiv@secondary@component} (solar units) => 1.0 solRad>

In [4]:

b.add_solver('estimator.ebai', ebai_method='knn', solver='ebai_knn')
b.run_solver('ebai_knn', solution='ebai_knn_solution')

# 2. Extraer parámetros propuestos
sol = b.get_solution('ebai_knn_solution')
params_to_adopt = sol.get_value('adopt_parameters')

# 3. [MANTENIDO] Filtrar ecosw y esinw para asegurar órbita 100% circular
params_filtered = [
    p for p in params_to_adopt
    if not ("ecosw" in p or "esinw" in p)
]

# 4. [MANTENIDO] Adoptar la solución
# PHOEBE actualizará 'requivsumfrac' y, gracias a nuestra configuración inicial, 
# modificará 'requiv@primary' dejando el contacto de la secundaria intacto.
b.adopt_solution('ebai_knn_solution', adopt_parameters=params_filtered, adopt_values=True)


C:\Users\darkm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator KNeighborsRegressor from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\darkm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


<ParameterSet: 19 parameters | contexts: dataset, component>

In [38]:
print("R1 (requiv@primary)=", b.get_value('requiv@primary@component'))
print("R2 (requiv@secondary)=", b.get_value('requiv@secondary@component'))

print("Sum (R1+R2)/a=", b.get_value('requivsumfrac@binary@component'))
print("Ratio R1/R2=", b.get_value('requivratio@binary@component'))

print("Teff primary=", b.get_value('teff@primary@component'))
print("Teff secondary=", b.get_value('teff@secondary@component'))

print("Incl=", b.get_value('incl@binary@orbit@component'))
print("Period=", b.get_value('period@binary'))

# Lóbulos de Roche
RL1 = b.get_value('requiv_max@primary@component')
RL2 = b.get_value('requiv_max@secondary@component')

print("Roche lobe 1:", RL1)
print("Roche lobe 2:", RL2)

# [NUEVO] Verificación rápida de la topología:
import numpy as np
print("\n--- Verificación Topología Semi-Separada ---")
if np.isclose(b.get_value('requiv@secondary@component'), RL2):
    print("ÉXITO: La estrella secundaria está llenando perfectamente su lóbulo de Roche.")
else:
    print("ERROR: La topología semi-separada se ha roto.")

R1 (requiv@primary)= 1.0
R2 (requiv@secondary)= 1.6994519588420407
Sum (R1+R2)/a= 0.29910399465803
Ratio R1/R2= 1.6994519588420407
Teff primary= 25700.0
Teff secondary= 23374.95361720044
Incl= 78.11938695015135
Period= 3.6788
Roche lobe 1: 2.342509738772878
Roche lobe 2: 1.6994519588420407

--- Verificación Topología Semi-Separada ---
ÉXITO: La estrella secundaria está llenando perfectamente su lóbulo de Roche.
